In [ ]:
# 12_rag_vector_index — rag.doc_chunks -> Databricks Vector Search (Delta Sync) index
# DE가 적재한 rag.doc_chunks(008/10) corpus 위에 DS가 검색 index를 만든다.
# 임베딩: databricks-gte-large-en (FM API). Delta Sync로 doc_chunks 변경을 자동 반영한다.
# 이 notebook은 index 생성/동기화만 담당한다. 질의 생성/LLM 생성은 13_rag_chain 이다.


In [ ]:
# Databricks Runtime 15.4 LTS may not include the Vector Search SDK by default.
# Keep this as a notebook-level guard for manual runs; job tasks also declare the PyPI library.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("databricks.vector_search") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "databricks-vectorsearch"])


In [ ]:
dbutils.widgets.text("catalog", "access_drift_dev")
dbutils.widgets.text("vs_endpoint", "access_drift_vs")
dbutils.widgets.text("embedding_model", "databricks-gte-large-en")
dbutils.widgets.text("index_name", "")

CATALOG = dbutils.widgets.get("catalog")
VS_ENDPOINT = dbutils.widgets.get("vs_endpoint")
EMBEDDING_MODEL = dbutils.widgets.get("embedding_model")
SOURCE_TABLE = f"{CATALOG}.rag.doc_chunks"
INDEX_NAME = dbutils.widgets.get("index_name") or f"{CATALOG}.rag.doc_chunks_index"
PRIMARY_KEY = "chunk_id"
EMBEDDING_SOURCE_COLUMN = "chunk_text"

print(f"source table : {SOURCE_TABLE}")
print(f"index name   : {INDEX_NAME}")
print(f"vs endpoint  : {VS_ENDPOINT}")
print(f"embedding    : {EMBEDDING_MODEL}")


In [ ]:
# 선결 조건: corpus가 존재하고 row가 있어야 하며, Delta Sync는 Change Data Feed가 켜져 있어야 한다.
chunk_count = spark.table(SOURCE_TABLE).count()
assert chunk_count > 0, f"{SOURCE_TABLE} 가 비어 있습니다. 10_rag_evidence_seed 를 먼저 실행하세요."

cdf_enabled = (
    spark.sql(f"SHOW TBLPROPERTIES {SOURCE_TABLE}")
    .filter("key = 'delta.enableChangeDataFeed'")
    .collect()
)
cdf_on = bool(cdf_enabled) and cdf_enabled[0]["value"].lower() == "true"
if not cdf_on:
    # 008 migration 이 켜지만, 기존 테이블이면 여기서 보정한다.
    spark.sql(f"ALTER TABLE {SOURCE_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    print("enabled Change Data Feed on source table")
print(f"source rows: {chunk_count}, CDF: on")


In [ ]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)

# Vector Search endpoint 확보 (없으면 생성).
existing_endpoints = {e["name"] for e in vsc.list_endpoints().get("endpoints", [])}
if VS_ENDPOINT not in existing_endpoints:
    print(f"creating endpoint {VS_ENDPOINT} ...")
    vsc.create_endpoint(name=VS_ENDPOINT, endpoint_type="STANDARD")

vsc.wait_for_endpoint(VS_ENDPOINT, verbose=True)
print(f"endpoint ready: {VS_ENDPOINT}")


In [ ]:
# Delta Sync index 생성 (없으면 생성, 있으면 재동기화).
existing_indexes = {
    i["name"] for i in vsc.list_indexes(name=VS_ENDPOINT).get("vector_indexes", [])
}

if INDEX_NAME in existing_indexes:
    print(f"index exists, triggering sync: {INDEX_NAME}")
    index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
    index.sync()
else:
    print(f"creating delta sync index: {INDEX_NAME}")
    index = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=INDEX_NAME,
        source_table_name=SOURCE_TABLE,
        pipeline_type="TRIGGERED",
        primary_key=PRIMARY_KEY,
        embedding_source_column=EMBEDDING_SOURCE_COLUMN,
        embedding_model_endpoint_name=EMBEDDING_MODEL,
    )

index.wait_until_ready(verbose=True)
print("index ready")


In [ ]:
# Smoke: 대표 NHI residual access 질의로 검색이 되는지 확인한다.
smoke_query = "퇴사자 owner의 App Registration active credential 조치 방법"
res = index.similarity_search(
    query_text=smoke_query,
    columns=["chunk_id", "doc_id", "doc_type", "source_title", "source_uri"],
    filters={"drift_type": "nhi_residual_access", "principal_kind": "nhi"},
    num_results=4,
)
rows = res["result"]["data_array"]
assert len(rows) > 0, "similarity_search 결과가 비었습니다. 임베딩/동기화 상태를 확인하세요."
for r in rows:
    print(r[:4])

summary = [
    "rag vector index smoke passed",
    f"index: {INDEX_NAME}",
    f"endpoint: {VS_ENDPOINT}",
    f"source rows: {chunk_count}",
    f"smoke hits: {len(rows)}",
]
dbutils.notebook.exit("\n".join(summary))
